# Improving Interpretability of Machine Learning Systems

Interpretability is the ability to understand and explain model predictions and decisions. It is essential for transparency, trust, accountability, debugging, and bias detection.

## Importance of Interpretability
- Transparency: Clarifies how decisions are made in critical applications.
- Trust: Increases user and stakeholder confidence in model outputs.
- Accountability: Supports compliance and ethical deployment.
- Debugging: Aids in diagnosing errors and improving models.
- Bias Detection: Helps identify and mitigate unfairness.

## Key Concepts and Techniques
- Model Transparency: Understand architecture, parameters, and decision logic. Linear models and decision trees are inherently interpretable.
- Feature Importance: Rank features by impact using permutation importance or mean decrease in impurity.
- Local vs Global Interpretability:
  - Local: Explain individual predictions (LIME, SHAP).
  - Global: Understand overall behavior (feature importance, partial dependence).
- Model-Agnostic Methods: Techniques applicable to any model (LIME, SHAP, permutation importance).
- Visual Explanations: Use plots, charts, heatmaps to make behaviors accessible.


In [ ]:
import sys
!{sys.executable} -m pip install -q numpy pandas scikit-learn matplotlib lime shap

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

iris = load_iris()
X = iris.data
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

## LIME (Local Interpretable Model-Agnostic Explanations)
Explains a single prediction by approximating the model locally with a simple surrogate. It perturbs inputs around the instance, queries the model, and fits an interpretable model to those local predictions.

Steps:
- Train the model
- Select an instance to explain
- Perturb data around the instance
- Fit a local surrogate model
- Interpret the surrogate

In [ ]:
import lime
import lime.lime_tabular

instance = X_test[0].reshape(1, -1)
explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    discretize_continuous=True
)
explanation = explainer.explain_instance(instance[0], model.predict_proba)
explanation.show_in_notebook(show_all=False)

## SHAP (SHapley Additive exPlanations)
Attributes a prediction to features using cooperative game theory. Measures the average marginal contribution of each feature, offering consistent local explanations and global importance views.

Steps:
- Train the model
- Compute SHAP values
- Visualize with summary and dependence plots

In [ ]:
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)
feature_names = np.array(iris.feature_names)
shap.summary_plot(shap_values, X_test, feature_names)
plt.tight_layout()
plt.show()

## Balancing Generalization and Interpretability
Complex models often generalize well but are harder to explain; simpler models are more interpretable. Balance both using:
- Model Simplification: Prefer simpler architectures or prune complex ones.
- Regularization: Control overfitting to improve stability and clarity.
- Ensembling: Combine models to maintain performance while adding interpretability views.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
tree_rules = export_text(tree, feature_names=iris.feature_names)
print(tree_rules)